In [1]:
# March Machine Learning Mania 2026
## Brier Score Optimized Probability Model

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

import lightgbm as lgb

import warnings
warnings.filterwarnings("ignore")

In [3]:
import os

DATA_PATH = "/kaggle/input/competitions/march-machine-learning-mania-2026"

def load_csv(filename):
    return pd.read_csv(os.path.join(DATA_PATH, filename))

M_regular = load_csv("MRegularSeasonCompactResults.csv")
M_tourney = load_csv("MNCAATourneyCompactResults.csv")
M_seeds = load_csv("MNCAATourneySeeds.csv")
M_teams = load_csv("MTeams.csv")

W_regular = load_csv("WRegularSeasonCompactResults.csv")
W_tourney = load_csv("WNCAATourneyCompactResults.csv")
W_seeds = load_csv("WNCAATourneySeeds.csv")
W_teams = load_csv("WTeams.csv")

In [4]:
regular = pd.concat([M_regular, W_regular], ignore_index=True)
tourney = pd.concat([M_tourney, W_tourney], ignore_index=True)
seeds = pd.concat([M_seeds, W_seeds], ignore_index=True)

In [5]:
    def prepare_games(df):
        df = df.copy()
        
        df["Team1"] = np.minimum(df["WTeamID"], df["LTeamID"])
        df["Team2"] = np.maximum(df["WTeamID"], df["LTeamID"])
        
        df["Team1_win"] = (df["WTeamID"] == df["Team1"]).astype(int)
        
        return df[["Season", "Team1", "Team2", "Team1_win"]]
    
    games = prepare_games(regular)

In [6]:
team_stats = regular.copy()

wins = team_stats.groupby(["Season", "WTeamID"]).size().reset_index(name="Wins")
losses = team_stats.groupby(["Season", "LTeamID"]).size().reset_index(name="Losses")

wins.columns = ["Season", "TeamID", "Wins"]
losses.columns = ["Season", "TeamID", "Losses"]

team_summary = pd.merge(wins, losses, on=["Season", "TeamID"], how="outer").fillna(0)

team_summary["WinPct"] = team_summary["Wins"] / (team_summary["Wins"] + team_summary["Losses"])

In [7]:
games = games.merge(team_summary, left_on=["Season", "Team1"], 
                    right_on=["Season", "TeamID"], how="left") \
             .rename(columns={"WinPct": "Team1_WinPct"}) \
             .drop(columns=["TeamID", "Wins", "Losses"])

games = games.merge(team_summary, left_on=["Season", "Team2"], 
                    right_on=["Season", "TeamID"], how="left") \
             .rename(columns={"WinPct": "Team2_WinPct"}) \
             .drop(columns=["TeamID", "Wins", "Losses"])

games["WinPct_Diff"] = games["Team1_WinPct"] - games["Team2_WinPct"]

In [8]:
train = games[games["Season"] <= 2021]
valid = games[games["Season"] > 2021]

features = ["WinPct_Diff"]

X_train = train[features]
y_train = train["Team1_win"]

X_valid = valid[features]
y_valid = valid["Team1_win"]

In [9]:
model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.01,
    max_depth=-1,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8
)

model.fit(X_train, y_train)

[LightGBM] [Info] Number of positive: 141373, number of negative: 145098
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003421 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 286471, number of used features: 1
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.493498 -> initscore=-0.026008
[LightGBM] [Info] Start training from score -0.026008


LGBMClassifier(colsample_bytree=0.8, learning_rate=0.01, n_estimators=1000,
               num_leaves=64, subsample=0.8)

In [10]:
valid_preds = model.predict_proba(X_valid)[:,1]

brier = mean_squared_error(y_valid, valid_preds)
print("Validation Brier Score:", brier)

Validation Brier Score: 0.16386472844269842


In [11]:
calibrator = CalibratedClassifierCV(model, method="isotonic", cv=5)
calibrator.fit(X_train, y_train)

valid_preds_cal = calibrator.predict_proba(X_valid)[:,1]

brier_cal = mean_squared_error(y_valid, valid_preds_cal)
print("Calibrated Brier Score:", brier_cal)

[LightGBM] [Info] Number of positive: 113098, number of negative: 116078
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001698 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 229176, number of used features: 1
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.493498 -> initscore=-0.026008
[LightGBM] [Info] Start training from score -0.026008
[LightGBM] [Info] Number of positive: 113098, number of negative: 116079
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001706 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 229177, number of used features: 1
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.493496 -> initscore=-0.026016
[LightGBM] [Info] Start training from score -0.026016
[LightGBM]

In [12]:
sample = load_csv("SampleSubmissionStage2.csv")

In [13]:
sub = sample.copy()

sub["Season"] = sub["ID"].str.split("_").str[0].astype(int)
sub["Team1"] = sub["ID"].str.split("_").str[1].astype(int)
sub["Team2"] = sub["ID"].str.split("_").str[2].astype(int)

In [14]:
sub = sub.merge(team_summary, 
                left_on=["Season", "Team1"], 
                right_on=["Season", "TeamID"], 
                how="left") \
         .rename(columns={"WinPct": "Team1_WinPct"}) \
         .drop(columns=["TeamID", "Wins", "Losses"])

sub = sub.merge(team_summary, 
                left_on=["Season", "Team2"], 
                right_on=["Season", "TeamID"], 
                how="left") \
         .rename(columns={"WinPct": "Team2_WinPct"}) \
         .drop(columns=["TeamID", "Wins", "Losses"])

sub["WinPct_Diff"] = sub["Team1_WinPct"] - sub["Team2_WinPct"]

In [15]:
X_sub = sub[features]

sub["Pred"] = calibrator.predict_proba(X_sub)[:, 1]

In [16]:
final_submission = sub[["ID", "Pred"]]

final_submission.to_csv("submission.csv", index=False)

final_submission.head()

,ID,Pred
0,2026_1101_1102,0.820435
1,2026_1101_1103,0.060503
2,2026_1101_1104,0.164821
3,2026_1101_1105,0.393938
4,2026_1101_1106,0.618639


In [17]:
len(sample), len(final_submission)

(132133, 132133)

In [18]:
sample = load_csv("SampleSubmissionStage2.csv")

sub = sample.copy()

sub["Season"] = sub["ID"].str.split("_").str[0].astype(int)
sub["Team1"] = sub["ID"].str.split("_").str[1].astype(int)
sub["Team2"] = sub["ID"].str.split("_").str[2].astype(int)

sub = sub.merge(team_summary, 
                left_on=["Season", "Team1"], 
                right_on=["Season", "TeamID"], 
                how="left") \
         .rename(columns={"WinPct": "Team1_WinPct"}) \
         .drop(columns=["TeamID", "Wins", "Losses"])

sub = sub.merge(team_summary, 
                left_on=["Season", "Team2"], 
                right_on=["Season", "TeamID"], 
                how="left") \
         .rename(columns={"WinPct": "Team2_WinPct"}) \
         .drop(columns=["TeamID", "Wins", "Losses"])

sub["WinPct_Diff"] = sub["Team1_WinPct"] - sub["Team2_WinPct"]

X_sub = sub[features]

sub["Pred"] = calibrator.predict_proba(X_sub)[:, 1]

# IMPORTANT: Preserve original order and all rows
submission = sample.copy()
submission["Pred"] = sub["Pred"]

submission.to_csv("submission.csv", index=False)

len(submission)

132133

In [19]:
sample = load_csv("SampleSubmissionStage2.csv")
len(sample)

132133

In [20]:
import os
os.listdir(DATA_PATH)

['Conferences.csv',
 'WNCAATourneyDetailedResults.csv',
 'WRegularSeasonCompactResults.csv',
 'MNCAATourneySeedRoundSlots.csv',
 'MRegularSeasonDetailedResults.csv',
 'MNCAATourneyCompactResults.csv',
 'MGameCities.csv',
 'WSecondaryTourneyCompactResults.csv',
 'WGameCities.csv',
 'MSeasons.csv',
 'WNCAATourneySlots.csv',
 'MSecondaryTourneyTeams.csv',
 'SampleSubmissionStage2.csv',
 'Cities.csv',
 'MTeamSpellings.csv',
 'MRegularSeasonCompactResults.csv',
 'MMasseyOrdinals.csv',
 'MSecondaryTourneyCompactResults.csv',
 'WTeams.csv',
 'WConferenceTourneyGames.csv',
 'MNCAATourneySlots.csv',
 'MNCAATourneySeeds.csv',
 'WNCAATourneyCompactResults.csv',
 'WSeasons.csv',
 'WNCAATourneySeeds.csv',
 'MTeamCoaches.csv',
 'MConferenceTourneyGames.csv',
 'WRegularSeasonDetailedResults.csv',
 'MNCAATourneyDetailedResults.csv',
 'WTeamSpellings.csv',
 'MTeamConferences.csv',
 'MTeams.csv',
 'WTeamConferences.csv',
 'SampleSubmissionStage1.csv',
 'WSecondaryTourneyTeams.csv']

In [21]:
sample1 = load_csv("SampleSubmissionStage1.csv")
sample2 = load_csv("SampleSubmissionStage2.csv")

print("Stage1 rows:", len(sample1))
print("Stage2 rows:", len(sample2))

Stage1 rows: 519144
Stage2 rows: 132133


In [22]:
sample = load_csv("SampleSubmissionStage1.csv")
print(len(sample))

519144


In [23]:
# Load correct submission template
sample = load_csv("SampleSubmissionStage1.csv")

sub = sample.copy()

# Parse ID
sub["Season"] = sub["ID"].str.split("_").str[0].astype(int)
sub["Team1"] = sub["ID"].str.split("_").str[1].astype(int)
sub["Team2"] = sub["ID"].str.split("_").str[2].astype(int)

# Merge features
sub = sub.merge(team_summary, 
                left_on=["Season", "Team1"], 
                right_on=["Season", "TeamID"], 
                how="left") \
         .rename(columns={"WinPct": "Team1_WinPct"}) \
         .drop(columns=["TeamID", "Wins", "Losses"])

sub = sub.merge(team_summary, 
                left_on=["Season", "Team2"], 
                right_on=["Season", "TeamID"], 
                how="left") \
         .rename(columns={"WinPct": "Team2_WinPct"}) \
         .drop(columns=["TeamID", "Wins", "Losses"])

sub["WinPct_Diff"] = sub["Team1_WinPct"] - sub["Team2_WinPct"]

X_sub = sub[features]

# Predict probabilities
sub["Pred"] = calibrator.predict_proba(X_sub)[:, 1]

# Final submission (preserve order)
submission = sample.copy()
submission["Pred"] = sub["Pred"]

submission.to_csv("submission.csv", index=False)

print(submission.shape)

(519144, 2)
